In [10]:
import numpy as np
import random
from collections import defaultdict

class GridWorld:
    """Simple 4x4 GridWorld environment for demonstration"""
    
    def __init__(self):
        self.grid_size = 4
        self.actions = ['up', 'down', 'left', 'right']
        self.terminal_states = [(0, 0), (3, 3)]  # Start and goal
        self.rewards = {(0, 0): 0, (3, 3): 1}  # Goal gives reward of 1
        
    def get_states(self):
        """Get all possible states"""
        return [(i, j) for i in range(self.grid_size) for j in range(self.grid_size)]
    
    def get_actions(self, state):
        """Get available actions for a state"""
        if state in self.terminal_states:
            return []
        return self.actions
    
    def step(self, state, action):
        """Take action in state and return (next_state, reward)"""
        if state in self.terminal_states:
            return state, 0
        
        i, j = state
        
        if action == 'up':
            next_state = (max(0, i-1), j)
        elif action == 'down':
            next_state = (min(self.grid_size-1, i+1), j)
        elif action == 'left':
            next_state = (i, max(0, j-1))
        elif action == 'right':
            next_state = (i, min(self.grid_size-1, j+1))
        
        reward = self.rewards.get(next_state, -0.1)  # Small negative reward for each step
        return next_state, reward
    
    def is_terminal(self, state):
        """Check if state is terminal"""
        return state in self.terminal_states
    
   

In [14]:

class MonteCarloAgent:
    """Monte Carlo Policy Iteration Agent"""
    
    def __init__(self, env, gamma=0.9, epsilon=0.1, seed=None):
        self.env = env
        self.gamma = gamma
        self.epsilon = epsilon

        if seed is not None:
            random.seed(seed)
            np.random.seed(seed)

        # Initialize Q-values, returns, and policy
        from collections import defaultdict
        self.Q = defaultdict(lambda: defaultdict(float))
        self.returns = defaultdict(lambda: defaultdict(list))
        self.policy = {}

        # Initialize random policy
        for state in env.get_states():
            if not env.is_terminal(state):
                self.policy[state] = random.choice(env.get_actions(state))

    
    def generate_episode(self, start_state=(1, 1), max_steps=100):
        """Generate an episode following current policy (ε-greedy)"""
        episode = []
        state = start_state
        steps = 0
        
        while not self.env.is_terminal(state) and steps < max_steps:
            # ε-greedy action
            if random.random() < self.epsilon:
                action = random.choice(self.env.get_actions(state))
            else:
                action = self.policy.get(state, random.choice(self.env.get_actions(state)))
            
            next_state, reward = self.env.step(state, action)
            episode.append((state, action, reward))
            state = next_state
            steps += 1
        
        return episode

    def monte_carlo_policy_evaluation(self, num_episodes=1000, start_state=(1, 1)):
        """First-visit Monte Carlo policy evaluation for the current policy"""
        for _ in range(num_episodes):
            episode = self.generate_episode(start_state=start_state)
            G = 0.0
            for t in reversed(range(len(episode))):
                state, action, reward = episode[t]
                G = self.gamma * G + reward
                # First-visit MC
                if not any(s == state and a == action for s, a, _ in episode[:t]):
                    self.returns[state][action].append(G)
                    self.Q[state][action] = float(np.mean(self.returns[state][action]))
        return self.get_state_values()

    def monte_carlo_policy_iteration(self, num_iterations=10, episodes_per_iteration=1000, start_state=(1, 1)):
        """Classic MC policy iteration: evaluate -> improve -> repeat"""
        for _ in range(num_iterations):
            # Policy Evaluation
            for _ in range(episodes_per_iteration):
                episode = self.generate_episode(start_state=start_state)
                G = 0.0
                for t in reversed(range(len(episode))):
                    state, action, reward = episode[t]
                    G = self.gamma * G + reward
                    if not any(s == state and a == action for s, a, _ in episode[:t]):
                        self.returns[state][action].append(G)
                        self.Q[state][action] = float(np.mean(self.returns[state][action]))
            
            # Policy Improvement (greedy with respect to Q)
            policy_stable = True
            for state in self.env.get_states():
                if not self.env.is_terminal(state):
                    old_action = self.policy.get(state)
                    actions = self.env.get_actions(state)
                    if actions:
                        best_action = max(actions, key=lambda a: self.Q[state][a])
                        self.policy[state] = best_action
                        if old_action != best_action:
                            policy_stable = False
            if policy_stable:
                break
        
        return self.policy, self.get_state_values()

    def get_state_values(self):
        """Return V(s) = max_a Q(s,a) for each non-terminal state, None for terminal"""
        V = {}
        for i in range(self.env.grid_size):
            for j in range(self.env.grid_size):
                state = (i, j)
                if self.env.is_terminal(state):
                    V[state] = None
                else:
                    actions = self.env.get_actions(state)
                    V[state] = max(self.Q[state][a] for a in actions) if actions else 0.0
        return V

In [15]:

if __name__ == "__main__":
    env = GridWorld()
    agent = MonteCarloAgent(env, gamma=0.9, epsilon=0.1, seed=0)
    policy, V = agent.monte_carlo_policy_iteration(num_iterations=10, episodes_per_iteration=2000, start_state=(1,1))
    
    # Pretty print policy
    symbols = {'up':'↑','down':'↓','left':'←','right':'→'}
    for i in range(env.grid_size):
        row = []
        for j in range(env.grid_size):
            s = (i,j)
            if env.is_terminal(s):
                row.append("TERM")
            else:
                row.append(symbols.get(policy.get(s,'?'),'?'))
        print(" ".join(row))
    
    # Print value table (rounded)
    print("\nState values (rounded):")
    for i in range(env.grid_size):
        row = []
        for j in range(env.grid_size):
            s = (i,j)
            v = V[s]
            row.append(" TERM " if v is None else f"{v:6.2f}")
        print(" ".join(row))


TERM ← ↓ ↓
↑ ← ↓ ↓
↑ → → ↓
↑ → → TERM

State values (rounded):
 TERM    0.00  -0.78  -0.78
  0.00  -0.14  -0.57   0.72
 -0.51  -0.17   0.72   1.00
 -0.81  -0.67   1.00  TERM 
